# 一、《消息面可用的接口》
## （1）Akshare：
###      1、个股研报  stock_research_report_em
###      2、千股千评——主力控盘机构参与度   stock_comment_detail_zlkp_jgcyd_em
###      3、千股千评——综合评价历史评分 stock_comment_detail_zhpj_lspf_em
###      4、千股千评——市场热度、用户stock_comment_detail_scrd_focus_em
###      5、个股新闻  stock_news_em
###      6、股票热度——雪球  stock_hot_follow_xq
###      7、股票热度——东财  stock_hot_rank_em
###      8、股票热度——历史趋势及粉丝特征  stock_hot_rank_detail_em
###      9、股票热度——互动平台/互动易  stock_irm_cninfo
###      10、股票热度——个股人气榜-实时变动  stock_hot_rank_detail_realtime_em
###      11、股票热度——热门关键词  stock_hot_keyword_em
###      12、股票热度——个股人气榜-最新排名stock_hot_rank_latest_em
###      13、股票热度——热搜股票stock_hot_search_baidu
###      14、股票热度——相关股票stock_hot_rank_relate_em
###      15、资讯数据——财经早餐-东财财富stock_info_cjzc_em
###      16、资讯数据——全球财经快讯-东财财富 stock_info_global_em
###      17、资讯数据——全球财经快讯-新浪财经 stock_info_global_sina
###      18、资讯数据——快讯-富途牛牛 stock_info_global_futu
###      19、资讯数据——全球财经直播-同花顺财经 stock_info_global_ths
###      20、资讯数据——电报-财联社 stock_info_global_cls
###      21、资讯数据——证券原创-新浪财经 stock_info_broker_sina


# 二、《资金面可用的接口》



# 三、《实时个股、市场行情可用的接口》
# （1）Akshare：
#       1、盘口异动  stock_changes_em
#       2、涨停板行情——涨停股池  stock_zt_pool_em
#       3、涨停板行情——昨日涨停股池  stock_zt_pool_previous_em
#       4、涨停板行情——涨停股池  stock_zt_pool_em
#       5、涨停板行情——强势股池  stock_zt_pool_strong_em
#       6、涨停板行情——次新股池  stock_zt_pool_sub_new_em
#       7、涨停板行情——炸板股池  stock_zt_pool_zbgc_em
#       8、涨停板行情——跌停股池  stock_zt_pool_dtgc_em



# （4）---------------------《个股基本面可用的接口》--------------------------------
# （1）Akshare：
#       1、


# （5）---------------------《个股量价面可用的接口》--------------------------------
# （1）Akshare：
#       1、大盘赚钱效应分析  stock_market_activity_legu
#       2、技术指标-创新高  stock_rank_cxg_ths
#       3、技术指标-创新低  stock_rank_cxd_ths
#       4、技术指标-连续上涨  stock_rank_lxsz_ths
#       5、技术指标-连续下跌  stock_rank_lxxd_ths
#       6、技术指标-持续放量  stock_rank_cxfl_ths
#       7、技术指标-持续缩量  stock_rank_cxsl_ths
#       8、技术指标-向上突破  stock_rank_xstp_ths
#       9、技术指标-向下突破  stock_rank_xxtp_ths
#      10、技术指标-量价齐升  stock_rank_ljqs_ths
#      11、技术指标-量价齐跌  stock_rank_ljqd_ths
#      12、技术指标-险资举牌  stock_rank_xzjp_ths

# 项目架构：

TRADESYSTEM
   |-----data（原始数据获取与存储 + 可能的据简单处理）
         |------个股量价面数据（1m/5m/日频/周频/月频等）
         |------个股基本面财务数据
         |------个股消息面新闻数据
         |------个股资金面数据
         |------个股当日实时行情
         |------......（更多）
   |-----preprocess（配合策略的数据预处理与特征工程）
   |-----strategy（各类因子与策略实现）
         |------strategy1 多因子模型数据预处理、因子构建、多因子打分选股及 三类回测（V1版）
         |------strategy2 多因子模型数据预处理、因子构建、多因子打分选股及 三类回测（V2优化版，各模块进行模块化处理，真实市场回测引擎）
         |------strategy3 多因子模型数据预处理、因子构建、OLS模型选股及 一类胜率回测（V3优化版，使用OLS模型而非打分，设置多种截面线性模型构建方式，用每次采样频率的未来horizon_n天内的预期最大收益率作为被解释变量）
         |------strategy4 基于图神经网络GNN预测个股收益率、波动率选股模型（基本面因子+量价因子，V1版本）
         |------strategy5 基于遗传算法的因子挖掘与评估回测、选取可用因子（V1版本）
   |-----backtrade（回测）

# 回测的三种类型（逐步深入）
- 第一类：纯粹回测，统计策略胜率模块（在回测时间段内，每次的选股是否带来正收益）
- 第二类：定期调仓换股的回测模块（根据策略频率，定期调仓换股，比如每周五、或者每个月30号），不考虑，或者弱考虑仓位、滑点、成本冲击、真实买/卖成交价、投资组合权重、投资组合动态优化调仓等（比如买/卖成交价可能就是当日收盘价，根据模型选股结果一次性调整组合中的所有标的而非个别）
- 第三类：贴近真实交易环境的回测模块，考虑仓位、滑点、成本冲击、真实买/卖成交价、投资组合权重、投资组合动态优化调仓等

# 模型预测类型
- 1、预测未来价格（时序RNN），评估：进一步计算收益率（日/周/月不同维度），取每期模型预测的收益率topN，回测胜率、IC
- 2、预测未来收益率、挖掘新的收益率alpha因子（多因子模型+打分、时序、深度/机器学习、transformeer等、图神经网络）。取每期模型预测的收益率topN，回测胜率、IC
- 3、预测未来波动率，进行投资组合动态优化（多因子、时序、深度/机器学习、transformeer等、图神经网络）
- 4、预测收益率+波动率，或者基于预测量构建新的得分因子